## TASK - 6
## Feature Selection and Recommendation Quality Evaluation

In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import os

In [2]:
#Step 2: Load existing cleaned dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (4005, 24)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,topics,related_talks,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,with the same humor and humanity he exuded in ...,thank you so much chris and it's truly a great...,"['alternative energy', 'cars', 'climate change...",averting the climate crisis alternative energy...
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you've ever seen,you've never seen data presented like this wit...,about 10 years ago i took on the task to teach...,"['africa', 'asia', 'google', 'demo', 'economic...",the best stats you've ever seen africa asia go...
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...,"['computers', 'entertainment', 'interface desi...",simplicity sells computers entertainment inter...
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,in an emotionally charged talk macarthur-winni...,if you're here today and i'm very happy that y...,"['macarthur grant', 'activism', 'business', 'c...",greening the ghetto macarthur grant activism b...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity,sir ken robinson makes an entertaining and pro...,good morning how are you audience good it's be...,"['children', 'creativity', 'culture', 'dance',...",do schools kill creativity children creativity...


In [3]:
#Step 3: Check the available columns
print(df.columns.tolist())

['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text']


In [4]:
# check
df[["title", "description", "transcript", "topics"]].head()

,title,description,transcript,topics
0,Averting the climate crisis,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...","['alternative energy', 'cars', 'climate change..."
1,The best stats you've ever seen,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...","['Africa', 'Asia', 'Google', 'demo', 'economic..."
2,Simplicity sells,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...","['computers', 'entertainment', 'interface desi..."
3,Greening the ghetto,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,"['MacArthur grant', 'activism', 'business', 'c..."
4,Do schools kill creativity?,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,"['children', 'creativity', 'culture', 'dance',..."


In [5]:
# step 4: Handle missing values
df["title"] = df["title"].fillna("")
df["description"] = df["description"].fillna("")
df["transcript"] = df["transcript"].fillna("")
df["topics"] = df["topics"].fillna("")

print("Missing values:")
print(df[["title", "description", "transcript", "topics"]].isnull().sum())


Missing values:
title          0
description    0
transcript     0
topics         0
dtype: int64


In [6]:
# step 5: Select five TED Talks for evaluation
selected_titles = [
    "Averting the climate crisis",
    "The radical promise of the multi-touch interface",
    "One Laptop per Child",
    "Do schools kill creativity?"
]

fifth_title = df.loc[
    ~df["title"].isin(selected_titles), "title"
].dropna().iloc[0]

selected_titles.append(fifth_title)

print("Selected TED Talks:")

for title in selected_titles:
    print("-", title)

Selected TED Talks:
- Averting the climate crisis
- The radical promise of the multi-touch interface
- One Laptop per Child
- Do schools kill creativity?
- The best stats you've ever seen


In [7]:
#Step 6: Create the four feature combinations
feature_combinations = {
    "Experiment 1 - Title Only":
        df["title"],

    "Experiment 2 - Title + Tags":
        df["title"] + " " + df["topics"],

    "Experiment 3 - Title + Description + Tags":
        df["title"] + " " +
        df["description"] + " " +
        df["topics"],

    "Experiment 4 - Title + Description + Transcript + Tags":
        df["title"] + " " +
        df["description"] + " " +
        df["transcript"] + " " +
        df["topics"]
}

print("Four feature combinations created successfully.")

Four feature combinations created successfully.


In [8]:
# Step 7: Create a recommendation function
def recommend_talks_by_features(title, similarity_matrix, top_n=5):

    if title not in df["title"].values:
        print("TED Talk not found:", title)
        return []

    idx = df.index[df["title"] == title][0]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in similarity_scores:
        if i != idx:
            recommendations.append({
                "Recommended Talk": df.iloc[i]["title"],
                "Similarity Score": round(float(score), 4)
            })

        if len(recommendations) == top_n:
            break

    return recommendations

In [9]:
# Step 8: Run all four experiments
comparison_results = []

for experiment_name, text_data in feature_combinations.items():

    print("\n", "=" * 70)
    print(experiment_name)
    print("=" * 70)

    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2
    )

    tfidf_matrix = tfidf.fit_transform(text_data)

    cosine_sim = cosine_similarity(tfidf_matrix)

    for title in selected_titles:

        recommendations = recommend_talks_by_features(
            title,
            cosine_sim,
            top_n=5
        )

        print("\nSelected Talk:", title)

        for recommendation in recommendations:

            print(recommendation["Recommended Talk"])

            comparison_results.append({
                "Experiment": experiment_name,
                "Selected Talk": title,
                "Recommended Talk":
                    recommendation["Recommended Talk"],
                "Similarity Score":
                    recommendation["Similarity Score"]
            })

    del tfidf_matrix
    del cosine_sim
    del tfidf


Experiment 1 - Title Only

Selected Talk: Averting the climate crisis
New thinking on the climate crisis
The post-crisis consumer
The global learning crisis -- and what to do about it
The state of the climate — and what we might do about it
Why you should be a climate activist

Selected Talk: The radical promise of the multi-touch interface
Massively multi-player… thumb-wrestling?
The technology of touch
A doctor's touch
The promise of research with stem cells
The best computer interface? Maybe ... your hands

Selected Talk: One Laptop per Child
One Laptop per Child, two years on
What it's like to be the child of immigrants
A child of the state
Teach every child about food
Teaching one child at a time

Selected Talk: Do schools kill creativity?
4 lessons in creativity
How state budgets are breaking US schools
Our failing schools. Enough is enough!
Fashion and creativity
Don't kill your language

Selected Talk: The best stats you've ever seen
The best kindergarten you’ve ever seen
The 

In [10]:
#Step 9: Create the comparison DataFrame
comparison_df = pd.DataFrame(comparison_results)

comparison_df.head(20)

,Experiment,Selected Talk,Recommended Talk,Similarity Score
0,Experiment 1 - Title Only,Averting the climate crisis,New thinking on the climate crisis,0.8128
1,Experiment 1 - Title Only,Averting the climate crisis,The post-crisis consumer,0.3202
2,Experiment 1 - Title Only,Averting the climate crisis,The global learning crisis -- and what to do a...,0.3006
3,Experiment 1 - Title Only,Averting the climate crisis,The state of the climate — and what we might d...,0.2925
4,Experiment 1 - Title Only,Averting the climate crisis,Why you should be a climate activist,0.2925
5,Experiment 1 - Title Only,The radical promise of the multi-touch interface,Massively multi-player… thumb-wrestling?,0.4659
6,Experiment 1 - Title Only,The radical promise of the multi-touch interface,The technology of touch,0.3459
7,Experiment 1 - Title Only,The radical promise of the multi-touch interface,A doctor's touch,0.3270
8,Experiment 1 - Title Only,The radical promise of the multi-touch interface,The promise of research with stem cells,0.2256
9,Experiment 1 - Title Only,The radical promise of the multi-touch interface,The best computer interface? Maybe ... your hands,0.2247


In [11]:
#Step 10: Compare recommendations for each experiment
for title in selected_titles:

    print("\nSelected Talk:", title)

    result = comparison_df[
        comparison_df["Selected Talk"] == title
    ]

    display(
        result.pivot(
            index="Experiment",
            columns=None,
            values="Recommended Talk"
        ) if False else
        result[[
            "Experiment",
            "Recommended Talk",
            "Similarity Score"
        ]]
    )


Selected Talk: Averting the climate crisis


,Experiment,Recommended Talk,Similarity Score
0,Experiment 1 - Title Only,New thinking on the climate crisis,0.8128
1,Experiment 1 - Title Only,The post-crisis consumer,0.3202
2,Experiment 1 - Title Only,The global learning crisis -- and what to do a...,0.3006
3,Experiment 1 - Title Only,The state of the climate — and what we might d...,0.2925
4,Experiment 1 - Title Only,Why you should be a climate activist,0.2925
25,Experiment 2 - Title + Tags,New thinking on the climate crisis,0.4893
26,Experiment 2 - Title + Tags,Why I must speak out about climate change,0.4004
27,Experiment 2 - Title + Tags,5 transformational policies for a prosperous a...,0.3856
28,Experiment 2 - Title + Tags,The science behind a climate headline,0.3837
29,Experiment 2 - Title + Tags,The Earth is full,0.3767



Selected Talk: The radical promise of the multi-touch interface


,Experiment,Recommended Talk,Similarity Score
5,Experiment 1 - Title Only,Massively multi-player… thumb-wrestling?,0.4659
6,Experiment 1 - Title Only,The technology of touch,0.3459
7,Experiment 1 - Title Only,A doctor's touch,0.3270
8,Experiment 1 - Title Only,The promise of research with stem cells,0.2256
9,Experiment 1 - Title Only,The best computer interface? Maybe ... your hands,0.2247
30,Experiment 2 - Title + Tags,Meet the SixthSense interaction,0.7307
31,Experiment 2 - Title + Tags,Why Google Glass?,0.5928
32,Experiment 2 - Title + Tags,Rethink the desktop with BumpTop,0.4315
33,Experiment 2 - Title + Tags,Reach into the computer and grab a pixel,0.4124
34,Experiment 2 - Title + Tags,"5 predictions, from 1984",0.4056



Selected Talk: One Laptop per Child


,Experiment,Recommended Talk,Similarity Score
10,Experiment 1 - Title Only,"One Laptop per Child, two years on",0.8883
11,Experiment 1 - Title Only,What it's like to be the child of immigrants,0.3634
12,Experiment 1 - Title Only,A child of the state,0.3038
13,Experiment 1 - Title Only,Teach every child about food,0.2786
14,Experiment 1 - Title Only,Teaching one child at a time,0.2667
35,Experiment 2 - Title + Tags,"One Laptop per Child, two years on",0.3298
36,Experiment 2 - Title + Tags,My journey into movies that matter,0.3226
37,Experiment 2 - Title + Tags,Africa's cheetahs versus hippos,0.2961
38,Experiment 2 - Title + Tags,You are the future of philanthropy,0.2944
39,Experiment 2 - Title + Tags,The why and how of effective altruism,0.2891



Selected Talk: Do schools kill creativity?


,Experiment,Recommended Talk,Similarity Score
15,Experiment 1 - Title Only,4 lessons in creativity,0.4247
16,Experiment 1 - Title Only,How state budgets are breaking US schools,0.4061
17,Experiment 1 - Title Only,Our failing schools. Enough is enough!,0.3949
18,Experiment 1 - Title Only,Fashion and creativity,0.3932
19,Experiment 1 - Title Only,Don't kill your language,0.3840
40,Experiment 2 - Title + Tags,Creative ways to get kids to thrive in school,0.4116
41,Experiment 2 - Title + Tags,Kids need structure,0.2804
42,Experiment 2 - Title + Tags,What adults can learn from kids,0.2300
43,Experiment 2 - Title + Tags,"Doodlers, unite!",0.2280
44,Experiment 2 - Title + Tags,Bring on the learning revolution!,0.2191



Selected Talk: The best stats you've ever seen


,Experiment,Recommended Talk,Similarity Score
20,Experiment 1 - Title Only,The best kindergarten you’ve ever seen,0.7665
21,Experiment 1 - Title Only,The sea we've hardly seen,0.4955
22,Experiment 1 - Title Only,Cooking as never seen before,0.3227
23,Experiment 1 - Title Only,Surprising stats about child carseats,0.3171
24,Experiment 1 - Title Only,Turning powerful stats into art,0.2693
45,Experiment 2 - Title + Tags,New insights on poverty,0.4907
46,Experiment 2 - Title + Tags,How not to be ignorant about the world,0.4654
47,Experiment 2 - Title + Tags,Let my dataset change your mindset,0.3985
48,Experiment 2 - Title + Tags,Medicine for the 99 percent,0.3002
49,Experiment 2 - Title + Tags,The global goals we've made progress on -- and...,0.2918


In [12]:
# step 11: Save the comparison results
os.makedirs("../data/processed", exist_ok=True)

comparison_df.to_csv(
    "../data/processed/task6_comparison_results.csv",
    index=False
)

print("Task 6 comparison results saved successfully.")

Task 6 comparison results saved successfully.


### Task 6 – Observations

**Experiment 1 – Title Only**

Only the title was used for generating recommendations. The results were based on limited textual information. Some recommendations may be related, but the system may miss talks discussing similar concepts with different titles.

**Experiment 2 – Title + Tags**

Adding topics provides additional information about the subject of each TED Talk. Recommendations become more focused on common topics and interests.

**Experiment 3 – Title + Description + Tags**

Including descriptions provides more context about the content of each talk. This helps the system identify talks that discuss similar concepts even when their titles are different.

**Experiment 4 – Title + Description + Transcript + Tags**

Including transcripts provides detailed information about the actual spoken content. However, transcripts can also introduce additional words that may not always improve recommendation relevance.

### Overall Evaluation

The four experiments demonstrate that feature selection plays an important role in content-based recommendation systems. Adding more features provides additional information, but more features do not automatically guarantee better recommendations.

The final feature combination should be selected based on the relevance and meaningfulness of the generated recommendations.

### Final Feature Selection

After comparing all four experiments, the selected feature combination is **Title + Description + Tags** (Experiment 3), provided it gives the best relevance in the evaluation.

This combination provides a balance between meaningful textual information and recommendation quality.

The title identifies the subject, the description provides additional context, and the tags help identify related topics.

Therefore, this combination is selected as the preferred feature set for the content-based recommendation model if supported by the observed results.